In [7]:
# ================================================================
# NOTEBOOK : nb_silver_store
# Read from  : bronze_lakehouse → bronze_store
# Write to   : silver_lakehouse → silver_store
# ================================================================


StatementMeta(, 69f20458-7bbd-4ea1-abf3-682146dde574, 9, Finished, Available, Finished, False)

In [8]:
from pyspark.sql import functions as F
from pyspark.sql.functions import trim,col,upper,round,datediff,current_date,initcap,to_date

#OR
#from pyspark.sql.functions import *

bronze_df = spark.sql("SELECT * FROM bronze_lakehouse.bronze_store")

print(f"[BRONZE]ROWS:{bronze_df.count()}")


display(bronze_df)

StatementMeta(, 69f20458-7bbd-4ea1-abf3-682146dde574, 10, Finished, Available, Finished, False)

[BRONZE]ROWS:50


SynapseWidget(Synapse.DataFrame, cb587ac3-57ce-444c-a611-7d0594e2a5a2)

In [9]:
# ── check Dedup on StoreID ───────────────────────────────────────
duplicate_store_id = (bronze_df.groupby("StoreID")
.count()
.filter(col("count") > 1)
)
duplicate_store_id.show()


StatementMeta(, 69f20458-7bbd-4ea1-abf3-682146dde574, 11, Finished, Available, Finished, False)

+-------+-----+
|StoreID|count|
+-------+-----+
+-------+-----+



In [17]:
from pyspark.sql.functions import col

 # ── Nulls ──────────────────────────────────────────────────
silver_df = (
    bronze_df
.filter(col("storeID").isNotNull())
.filter(col("City").isNotNull())
.dropDuplicates(["StoreID"])
.withColumn("OpenDate",to_date(col("OpenDate"),"yyyy-MM-dd"))
.withColumn("SquareFootage",col("SquareFootage").cast("integer"))


 # ── Dedup on StoreID if exist ───────────────────────────────────────

 # ── Fix data types ─────────────────────────────────────────
  
 # ── Standardise strings ────────────────────────────────────
 .withColumn("StoreID",    upper(trim(col("StoreID"))))
 .withColumn("City",       initcap(trim(col("City"))))
 .withColumn("State",      initcap(trim(col("State"))))
 .withColumn("Region",    initcap(trim(col("Region"))))
 .withColumn("StoreType", initcap(trim(col("StoreType"))))

# ── Derived columns ────────────────────────────────────────
#Since How many years has this store open?
.withColumn("storagAgeYears", round(datediff(current_date(),col("OpenDate"))/365.25,1))

# Store size bucket for grouping in reports
.withColumn("SizeBucket",

F.when(col("squareFootage") < 3000, "Small")
.when(col("SquareFootage") < 7000, "Medium")
.otherwise("Large"))

#___________ Is this a high-value store type?

.withColumn(("IsPremiumStore"),col("StoreType").isin(["Premium", "Flagship"]))

# ── Metadata ───────────────────────────────────────────────
.withColumn("silver_loads_ts",F.current_timestamp())
)

silver_df.write.format("delta").mode("overwrite") \
.option("overwriteSchema", "True").saveAsTable("silver_store")

print(f"[Done] Silver_store_written: {silver_df.count()} rows")
display(silver_df)

StatementMeta(, 69f20458-7bbd-4ea1-abf3-682146dde574, 20, Finished, Available, Finished, False)

[Done] Silver_store_written: 50 rows


SynapseWidget(Synapse.DataFrame, aeeb2645-b273-4850-baee-ff024b048117)

In [21]:
bronze_df.schema["SquareFootage"].dataType

StatementMeta(, b7ae11b1-c31d-4bfd-9af5-156b1edbddce, 41, Finished, Available, Finished, False)

IntegerType()